# Fusion Retrieval in Document Search

## Overview

This code implements a Fusion Retrieval system that combines vector-based similarity search with keyword-based BM25 retrieval. The approach aims to leverage the strengths of both methods to improve the overall quality and relevance of document retrieval.

## Motivation

Traditional retrieval methods often rely on either semantic understanding (vector-based) or keyword matching (BM25). Each approach has its strengths and weaknesses. Fusion retrieval aims to combine these methods to create a more robust and accurate retrieval system that can handle a wider range of queries effectively.

## Key Components

1. PDF processing and text chunking
2. Vector store creation using FAISS and OpenAI embeddings
3. BM25 index creation for keyword-based retrieval
4. Custom fusion retrieval function that combines both methods

## Method Details

### Document Preprocessing

1. The PDF is loaded and split into chunks using RecursiveCharacterTextSplitter.
2. Chunks are cleaned by replacing 't' with spaces (likely addressing a specific formatting issue).

### Vector Store Creation

1. OpenAI embeddings are used to create vector representations of the text chunks.
2. A FAISS vector store is created from these embeddings for efficient similarity search.

### BM25 Index Creation

1. A BM25 index is created from the same text chunks used for the vector store.
2. This allows for keyword-based retrieval alongside the vector-based method.

### Fusion Retrieval Function

The `fusion_retrieval` function is the core of this implementation:

1. It takes a query and performs both vector-based and BM25-based retrieval.
2. Scores from both methods are normalized to a common scale.
3. A weighted combination of these scores is computed (controlled by the `alpha` parameter).
4. Documents are ranked based on the combined scores, and the top-k results are returned.

## Benefits of this Approach

1. Improved Retrieval Quality: By combining semantic and keyword-based search, the system can capture both conceptual similarity and exact keyword matches.
2. Flexibility: The `alpha` parameter allows for adjusting the balance between vector and keyword search based on specific use cases or query types.
3. Robustness: The combined approach can handle a wider range of queries effectively, mitigating weaknesses of individual methods.
4. Customizability: The system can be easily adapted to use different vector stores or keyword-based retrieval methods.

## Conclusion

Fusion retrieval represents a powerful approach to document search that combines the strengths of semantic understanding and keyword matching. By leveraging both vector-based and BM25 retrieval methods, it offers a more comprehensive and flexible solution for information retrieval tasks. This approach has potential applications in various fields where both conceptual similarity and keyword relevance are important, such as academic research, legal document search, or general-purpose search engines.

# Package Installation and Imports

The cell below installs all necessary packages required to run this notebook.


In [ ]:
# Install required packages
#!uv pip install langchain langchain-nvidia-ai-endpoints langchain-community langchain-text-splitters faiss-cpu numpy python-dotenv rank-bm25 pymupdf

In [1]:
!uv pip install numpy rank-bm25 pymupdf deepeval

Checked 4 packages in 28ms


In [2]:
import os
import sys
from dotenv import load_dotenv
from langchain_community.docstore.document import Document

from typing import List
from rank_bm25 import BM25Okapi
import numpy as np


from utils.helper_functions import *
from utils.evaluate_rag import *

# Load environment variables from a .env file
load_dotenv()

# Set the OpenAI API key environment variable
os.environ["NVIDIA_API_KEY"] = os.getenv('NVIDIA_API_KEY')

### Define document path

In [3]:
path = "pdfs/Understanding_Climate_Change.pdf"

### Encode the pdf to vector store and return split document from the step before to create BM25 instance

In [4]:
def encode_pdf_and_get_split_documents(path, chunk_size=1000, chunk_overlap=200):
    """
    Encodes a PDF book into a vector store using NVIDIA NIM embeddings.

    Args:
        path: The path to the PDF file.
        chunk_size: The desired size of each text chunk.
        chunk_overlap: The amount of overlap between consecutive chunks.

    Returns:
        A tuple of (FAISS vector store, cleaned text documents).
    """
    from langchain_community.document_loaders import PyPDFLoader
    from langchain_text_splitters import RecursiveCharacterTextSplitter
    from langchain_nvidia_ai_endpoints import NVIDIAEmbeddings
    from langchain_community.vectorstores import FAISS

    # Load PDF documents
    loader = PyPDFLoader(path)
    documents = loader.load()

    # Split documents into chunks
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, chunk_overlap=chunk_overlap, length_function=len
    )
    texts = text_splitter.split_documents(documents)
    cleaned_texts = replace_t_with_space(texts)

    # Create embeddings and vector store
    embeddings = NVIDIAEmbeddings()
    vectorstore = FAISS.from_documents(cleaned_texts, embeddings)

    return vectorstore, cleaned_texts

### Create vectorstore and get the chunked documents

In [7]:
vectorstore, cleaned_texts = encode_pdf_and_get_split_documents(path)

### Create a bm25 index for retrieving documents by keywords

In [9]:
from rank_bm25 import BM25Okapi

# Initalizing
corpus = [
    "Hello there good man!",
    "It is quite windy in London",
    "How is the weather today?"
]

tokenized_corpus = [doc.split(" ") for doc in corpus]

bm25 = BM25Okapi(tokenized_corpus)

# Ranking of documents
query = "windy London"
tokenized_query = query.split(" ")

doc_scores = bm25.get_scores(tokenized_query)
print(doc_scores)

doc_top_n = bm25.get_top_n(tokenized_query, corpus, n=1)
print(doc_top_n)

[0.         0.93729472 0.        ]
['It is quite windy in London']


In [10]:
tokenized_corpus

[['Hello', 'there', 'good', 'man!'],
 ['It', 'is', 'quite', 'windy', 'in', 'London'],
 ['How', 'is', 'the', 'weather', 'today?']]

In [11]:
def create_bm25_index(documents: List[Document]) -> BM25Okapi:
    """
    Create a BM25 index from the given documents.

    BM25 (Best Matching 25) is a ranking function used in information retrieval.
    It's based on the probabilistic retrieval framework and is an improvement over TF-IDF.

    Args:
    documents (List[Document]): List of documents to index.

    Returns:
    BM25Okapi: An index that can be used for BM25 scoring.
    """
    # Tokenize each document by splitting on whitespace
    # This is a simple approach and could be improved with more sophisticated tokenization
    tokenized_docs = [doc.page_content.split() for doc in documents]
    return BM25Okapi(tokenized_docs)

In [12]:
bm25 = create_bm25_index(cleaned_texts) # Create BM25 index from the cleaned texts (chunks)

### Define a function that retrieves both semantically and by keyword, normalizes the scores and gets the top k documents

In [13]:
def fusion_retrieval(vectorstore, bm25, query: str, k: int = 5, alpha: float = 0.5) -> List[Document]:
    """
    Perform fusion retrieval combining keyword-based (BM25) and vector-based search.

    Args:
    vectorstore (VectorStore): The vectorstore containing the documents.
    bm25 (BM25Okapi): Pre-computed BM25 index.
    query (str): The query string.
    k (int): The number of documents to retrieve.
    alpha (float): The weight for vector search scores (1-alpha will be the weight for BM25 scores).

    Returns:
    List[Document]: The top k documents based on the combined scores.
    """
    
    epsilon = 1e-8

    # Step 1: Get all documents from the vectorstore
    # Get total doc count from the FAISS index
    n_docs = vectorstore.index.ntotal

    # Step 1: Get all documents via vector search using the query
    all_docs_with_scores = vectorstore.similarity_search_with_score(query, k=n_docs)
    all_docs = [doc for doc, _ in all_docs_with_scores]

    # Step 2: Perform BM25 search
    bm25_scores = bm25.get_scores(query.split())

    # Step 3: Perform vector search
    vector_results = all_docs_with_scores  # Already retrieved above
    
    # Step 4: Normalize scores
    vector_scores = np.array([score for _, score in vector_results])
    vector_scores = 1 - (vector_scores - np.min(vector_scores)) / (np.max(vector_scores) - np.min(vector_scores) + epsilon)

    bm25_scores = (bm25_scores - np.min(bm25_scores)) / (np.max(bm25_scores) -  np.min(bm25_scores) + epsilon)

    # Step 5: Combine scores
    combined_scores = alpha * vector_scores + (1 - alpha) * bm25_scores  

    # Step 6: Rank documents
    sorted_indices = np.argsort(combined_scores)[::-1]
    
    # Step 7: Return top k documents
    return [all_docs[i] for i in sorted_indices[:k]]

### Use Case example

In [14]:
# Query
query = "What are the impacts of climate change on the environment?"

# Perform fusion retrieval
top_docs = fusion_retrieval(vectorstore, bm25, query, k=5, alpha=0.8)
docs_content = [doc.page_content for doc in top_docs]
show_context(docs_content)

Context 1:
The Arctic is warming at more than twice the global average rate, leading to significant ice 
loss. Antarctic ice sheets are also losing mass, contributing to sea level rise. This melting 
affects global ocean currents and weather patterns. 
Glacial Retreat 
Glaciers around the world are retreating, affecting water supplies for millions of people. 
Regions dependent on glacial meltwater, such as the Himalayas and the Andes, face 
particular risks. Glacial melt also impacts hydropower generation and agriculture. 
Coastal Erosion 
Rising sea levels and increased storm surges are accelerating coastal erosion, threatening 
homes, infrastructure, and ecosystems. Low-lying islands and coastal regions are especially 
vulnerable. Coastal communities must invest in adaptation measures like sea walls and 
managed retreats. 
Extreme Weather Events 
Climate change is linked to an increase in the frequency and severity of extreme weather


Context 2:
development of eco-friendly fertilize

## Langchain Implementation

In [17]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_classic.retrievers.ensemble import EnsembleRetriever
from langchain_community.retrievers import BM25Retriever
from langchain_nvidia_ai_endpoints import ChatNVIDIA, NVIDIAEmbeddings
from langchain_classic.chains import RetrievalQA
from langchain_core.prompts import PromptTemplate

# Load PDF
loader = PyPDFLoader("pdfs/Understanding_Climate_Change.pdf")
documents = loader.load()

# Split into chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=512,
    chunk_overlap=50
)
splits = text_splitter.split_documents(documents)

# Dense retriever (semantic)
embeddings = NVIDIAEmbeddings(model="nvidia/nv-embedqa-e5-v5")
vectorstore = Chroma.from_documents(splits, embeddings)
dense_retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# Sparse retriever (keyword - BM25)
sparse_retriever = BM25Retriever.from_documents(splits)
sparse_retriever.k = 3

alpha: float = 0.5

# Hybrid retriever
hybrid_retriever = EnsembleRetriever(
    retrievers=[dense_retriever, sparse_retriever],
    weights=[alpha, 1 - alpha]
)

# LLM
llm = ChatNVIDIA(model="meta/llama-3.3-70b-instruct", temperature=0)

# Create RAG chain
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=hybrid_retriever,
    return_source_documents=True
)

# Query with generation
query = "What is the main topic of this document?"
result = qa_chain.invoke({"query": query})

print("Answer:", result["result"])
print("\nSources:")
for doc in result["source_documents"]:
    print(f"- {doc.page_content[:200]}...")

Answer: The main topic of this document appears to be climate change, its causes, effects, and the importance of sustainability and collective action to address it.

Sources:
- health and sustainability of our planet for generations to come. Together, we can create a 
resilient, equitable, and thriving world....
- emissions, particularly methane, which is a potent greenhouse gas. Innovations in fracking 
technology have made natural gas more accessible, but this comes with environmental and 
health concerns. 
D...
- equitable access to climate adaptation resources are essential....
- for life on Earth, as it keeps the planet warm enough to support life. However, human 
activities have intensified this natural process, leading to a warmer climate. 
Fossil Fuels 
Burning fossil fuel...
- specific contexts and engaging communities in climate action....
- challenges. This vision includes healthy ecosystems, sustainable economies, and social 
justice. Achieving this vision requires commitme

In [18]:
# Hybrid retriever
alpha = 0.1
hybrid_retriever = EnsembleRetriever(
    retrievers=[dense_retriever, sparse_retriever],
    weights=[alpha, 1 - alpha]
)

# LLM
llm = ChatNVIDIA(model="meta/llama-3.3-70b-instruct", temperature=0)

# Create RAG chain
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=hybrid_retriever,
    return_source_documents=True
)

# Query with generation
query = "hat are the impacts of climate change?"
result = qa_chain.invoke({"query": query})

print("Answer:", result["result"])
print("\nSources:")
for doc in result["source_documents"]:
    print(f"- {doc.page_content[:200]}...")

Answer: The impacts of climate change include:

1. Rising Temperatures: Global temperatures have risen by about 1.2 degrees Celsius (2.2 degrees Fahrenheit) since the late 19th century.
2. Increased frequency and severity of extreme weather events such as heatwaves, hurricanes, and floods.
3. Economic disruption: Climate-related disasters can cause significant economic disruption.
4. Social and Environmental Costs: Climate change exacerbates social inequalities, with marginalized communities often bearing the brunt of its impacts.
5. Environmental costs: Loss of biodiversity, ecosystem degradation, and decreased availability of natural resources.

These impacts are not evenly distributed and often disproportionately affect vulnerable populations, including low-income communities, indigenous peoples, and marginalized groups.

Sources:
- impacts are not evenly distributed. Vulnerable populations, including low-income 
communities, indigenous peoples, and marginalized groups, often face t

## Direct ChromaDB Integration

- Pinecone
- Weviate

In [21]:
import chromadb
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer

from langchain_nvidia_ai_endpoints import ChatNVIDIA 

# Load PDF
loader = PyPDFLoader("pdfs/Understanding_Climate_Change.pdf")
documents = loader.load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=512, chunk_overlap=50)
chunks = text_splitter.split_documents(documents)

# Setup ChromaDB
client = chromadb.PersistentClient(path="./chroma_db")
collection = client.get_or_create_collection("pdf_docs")

# Embeddings
model = SentenceTransformer('all-MiniLM-L6-v2')

# Add documents
for idx, chunk in enumerate(chunks):
    collection.add(
        ids=[f"doc_{idx}"],
        documents=[chunk.page_content],
        embeddings=[model.encode(chunk.page_content).tolist()],
        metadatas=[{"page": chunk.metadata.get("page", 0)}]
    )

# Hybrid Query
query = "What are the main topics?"
query_emb = model.encode(query)

results = collection.query(
    query_embeddings=[query_emb.tolist()],
    n_results=5
)

# LLM Generation
llm = ChatNVIDIA(model="nvidia/nemotron-3-super-120b-a12b")
context = "\n\n".join(results['documents'][0])
prompt = f"Context:\n{context}\n\nQuestion: {query}\n\nAnswer:"
response = llm.invoke(prompt)

print("Answer:", response.content)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Answer: Based on the provided context, the **main topics** discussed are:

1.  **Community Engagement and Participation** (emphasized as essential for successful implementation, including specific contexts, urban initiatives like sustainable transportation/green infrastructure, and community-based conservation).
2.  **Interdisciplinary Approaches** (collaboration between scientists, policymakers, businesses, and communities for holistic and effective climate solutions).
3.  **Citizen Science** (engaging the public in scientific research and data collection to empower individuals in climate knowledge and action).
4.  **Climate and Health Impacts** (studying links between climate and health, developing technologies/treatments, improving health data systems to inform evidence-based policies).
5.  **Climate Education and Advocacy** (integrating climate change into educational curricula at schools, colleges, and universities to raise awareness and build knowledge, as highlighted in Chapter 

### RRF

In [22]:
!uv pip install scikit-learn sentence_transformers

Checked 2 packages in 36ms


In [23]:
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

def reciprocal_rank_fusion(rankings, k=60, weights=None):
    """RRF implementation"""
    if weights is None:
        weights = [1.0] * len(rankings)
    
    doc_scores = {}
    for ranking_idx, ranking in enumerate(rankings):
        weight = weights[ranking_idx]
        for rank_position, doc_id in enumerate(ranking, start=1):
            rrf_score = weight * (1.0 / (k + rank_position))
            doc_scores[doc_id] = doc_scores.get(doc_id, 0.0) + rrf_score
    
    return sorted(doc_scores.items(), key=lambda x: x[1], reverse=True)


# Sample documents
documents = [
    "Machine learning is a subset of artificial intelligence",
    "Deep learning uses neural networks with multiple layers",
    "Python is popular for data science and ML",
    "Natural language processing helps computers understand text",
    "Computer vision enables machines to interpret images"
]

# Query
query = "What is AI"

# ============================================
# DENSE SEARCH (Vector/Semantic)
# ============================================
dense_model = SentenceTransformer('all-MiniLM-L6-v2')
doc_embeddings = dense_model.encode(documents)
query_embedding = dense_model.encode(query)

# Calculate cosine similarity
similarities = np.dot(doc_embeddings, query_embedding) / (
    np.linalg.norm(doc_embeddings, axis=1) * np.linalg.norm(query_embedding)
)

# Get ranked document IDs (by similarity)
dense_ranking = np.argsort(similarities)[::-1].tolist()

# ============================================
# SPARSE SEARCH (Keyword/BM25-like)
# ============================================
tfidf = TfidfVectorizer()
doc_tfidf = tfidf.fit_transform(documents)
query_tfidf = tfidf.transform([query])

# Calculate TF-IDF scores
tfidf_scores = (doc_tfidf @ query_tfidf.T).toarray().flatten()

# Get ranked document IDs
sparse_ranking = np.argsort(tfidf_scores)[::-1].tolist()

# ============================================
# HYBRID SEARCH WITH RRF
# ============================================
hybrid_results = reciprocal_rank_fusion(
    rankings=[dense_ranking, sparse_ranking],
    k=60,
    weights=[2.0, 1.0]  # Dense search more important
)

# ============================================
# DISPLAY RESULTS
# ============================================
print("QUERY:", query)
print("\n" + "="*60)
print("DENSE SEARCH RANKING:")
for rank, doc_idx in enumerate(dense_ranking, 1):
    print(f"{rank}. [{doc_idx}] {documents[doc_idx]}")

print("\n" + "="*60)
print("SPARSE SEARCH RANKING:")
for rank, doc_idx in enumerate(sparse_ranking, 1):
    print(f"{rank}. [{doc_idx}] {documents[doc_idx]}")

print("\n" + "="*60)
print("HYBRID RRF RESULTS:")
print("="*60)
for doc_idx, score in hybrid_results[:3]:  # Top 3
    print(f"Score: {score:.6f}")
    print(f"Doc:   {documents[doc_idx]}")
    print()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

QUERY: What is AI

DENSE SEARCH RANKING:
1. [0] Machine learning is a subset of artificial intelligence
2. [4] Computer vision enables machines to interpret images
3. [3] Natural language processing helps computers understand text
4. [1] Deep learning uses neural networks with multiple layers
5. [2] Python is popular for data science and ML

SPARSE SEARCH RANKING:
1. [0] Machine learning is a subset of artificial intelligence
2. [2] Python is popular for data science and ML
3. [4] Computer vision enables machines to interpret images
4. [3] Natural language processing helps computers understand text
5. [1] Deep learning uses neural networks with multiple layers

HYBRID RRF RESULTS:
Score: 0.049180
Doc:   Machine learning is a subset of artificial intelligence

Score: 0.048131
Doc:   Computer vision enables machines to interpret images

Score: 0.047371
Doc:   Natural language processing helps computers understand text



In [24]:
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

def relative_score_fusion(score_lists, weights=None):
    """
    RSF implementation - uses actual scores, not just ranks
    
    Args:
        score_lists: List of lists, each containing (doc_id, score) tuples
        weights: Optional weights for each retriever
    """
    if weights is None:
        weights = [1.0] * len(score_lists)
    
    doc_scores = {}
    
    for retriever_idx, scores in enumerate(score_lists):
        weight = weights[retriever_idx]
        
        # Extract just the scores for normalization
        raw_scores = [score for _, score in scores]
        
        if len(raw_scores) == 0:
            continue
            
        min_score = min(raw_scores)
        max_score = max(raw_scores)
        score_range = max_score - min_score
        
        # Normalize and accumulate
        for doc_id, score in scores:
            if score_range == 0:
                normalized = 1.0  # All scores equal
            else:
                normalized = (score - min_score) / score_range
            
            doc_scores[doc_id] = doc_scores.get(doc_id, 0.0) + (weight * normalized)
    
    return sorted(doc_scores.items(), key=lambda x: x[1], reverse=True)


# Sample documents
documents = [
    "Machine learning is a subset of artificial intelligence",
    "Deep learning uses neural networks with multiple layers",
    "Python is popular for data science and ML",
    "Natural language processing helps computers understand text",
    "Computer vision enables machines to interpret images"
]

# Query
query = "What is AI"

# ============================================
# DENSE SEARCH (Vector/Semantic)
# ============================================
dense_model = SentenceTransformer('all-MiniLM-L6-v2')
doc_embeddings = dense_model.encode(documents)
query_embedding = dense_model.encode(query)

# Calculate cosine similarity
dense_scores = np.dot(doc_embeddings, query_embedding) / (
    np.linalg.norm(doc_embeddings, axis=1) * np.linalg.norm(query_embedding)
)

# Create (doc_id, score) tuples - sorted by score descending
dense_with_scores = [(i, float(dense_scores[i])) for i in range(len(documents))]
dense_with_scores.sort(key=lambda x: x[1], reverse=True)

# ============================================
# SPARSE SEARCH (Keyword/BM25-like)
# ============================================
tfidf = TfidfVectorizer()
doc_tfidf = tfidf.fit_transform(documents)
query_tfidf = tfidf.transform([query])

# Calculate TF-IDF scores
sparse_scores = (doc_tfidf @ query_tfidf.T).toarray().flatten()

# Create (doc_id, score) tuples - sorted by score descending
sparse_with_scores = [(i, float(sparse_scores[i])) for i in range(len(documents))]
sparse_with_scores.sort(key=lambda x: x[1], reverse=True)

# ============================================
# HYBRID SEARCH WITH RSF
# ============================================
hybrid_results = relative_score_fusion(
    score_lists=[dense_with_scores, sparse_with_scores],
    weights=[2.0, 1.0]  # Dense search more important
)

# ============================================
# DISPLAY RESULTS
# ============================================
print("QUERY:", query)

print("\n" + "="*60)
print("DENSE SEARCH (with raw scores):")
print("="*60)
for doc_idx, score in dense_with_scores:
    print(f"Score: {score:.4f} | [{doc_idx}] {documents[doc_idx]}")

print("\n" + "="*60)
print("SPARSE SEARCH (with raw scores):")
print("="*60)
for doc_idx, score in sparse_with_scores:
    print(f"Score: {score:.4f} | [{doc_idx}] {documents[doc_idx]}")

print("\n" + "="*60)
print("HYBRID RSF RESULTS:")
print("="*60)

# Show normalization breakdown for top results
print("\nDetailed scoring breakdown:\n")

# Recalculate normalized scores for display
dense_raw = [s for _, s in dense_with_scores]
sparse_raw = [s for _, s in sparse_with_scores]

dense_min, dense_max = min(dense_raw), max(dense_raw)
sparse_min, sparse_max = min(sparse_raw), max(sparse_raw)

for doc_idx, final_score in hybrid_results[:3]:
    dense_score = dense_scores[doc_idx]
    sparse_score = sparse_scores[doc_idx]
    
    # Normalized scores
    dense_norm = (dense_score - dense_min) / (dense_max - dense_min)
    sparse_norm = (sparse_score - sparse_min) / (sparse_max - sparse_min) if (sparse_max - sparse_min) > 0 else 0
    
    print(f"Doc [{doc_idx}]: {documents[doc_idx]}")
    print(f"  Dense:  raw={dense_score:.4f} → norm={dense_norm:.4f} × weight=2.0 = {dense_norm*2:.4f}")
    print(f"  Sparse: raw={sparse_score:.4f} → norm={sparse_norm:.4f} × weight=1.0 = {sparse_norm*1:.4f}")
    print(f"  Final RSF Score: {final_score:.4f}")
    print()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

QUERY: What is AI

DENSE SEARCH (with raw scores):
Score: 0.5238 | [0] Machine learning is a subset of artificial intelligence
Score: 0.3093 | [4] Computer vision enables machines to interpret images
Score: 0.2978 | [3] Natural language processing helps computers understand text
Score: 0.2312 | [1] Deep learning uses neural networks with multiple layers
Score: 0.2117 | [2] Python is popular for data science and ML

SPARSE SEARCH (with raw scores):
Score: 0.3214 | [0] Machine learning is a subset of artificial intelligence
Score: 0.2917 | [2] Python is popular for data science and ML
Score: 0.0000 | [1] Deep learning uses neural networks with multiple layers
Score: 0.0000 | [3] Natural language processing helps computers understand text
Score: 0.0000 | [4] Computer vision enables machines to interpret images

HYBRID RSF RESULTS:

Detailed scoring breakdown:

Doc [0]: Machine learning is a subset of artificial intelligence
  Dense:  raw=0.5238 → norm=1.0000 × weight=2.0 = 2.0000
  Sparse